# Econometric Analysis of Big-5 Football Player Valuations
**Repository:** `big5-player-valuation-econometrics`  
**Scope:** Data cleaning, exploratory data analysis, OLS regression modeling with HC1 robust standard errors, and diagnostic testing.

## 1. Environment Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Set plot styling
sns.set_theme(style="whitegrid")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# Load cleaned data from data/processed/
data_path = 'data/processed/cleaned_players_data.csv'
try:
    df = pd.read_csv(data_path)
    print(f"Data loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns")
except FileNotFoundError:
    print("File not found at data/processed/cleaned_players_data.csv. Please ensure data is placed correctly.")

## 2. Feature Engineering & Transformations
- **Log Transformation:** $\ln(\text{Market Value})$ to correct right-skewness and stabilize variance.
- **Quadratic Age Term:** $\text{Age}^2$ to capture non-linear parabolic valuation trajectory.

In [ ]:
if 'df' in locals():
    # Log market value
    df['log_market_value'] = np.log(df['market_value_eur'])
    
    # Age squared
    df['age_sq'] = df['age'] ** 2
    
    print(df[['market_value_eur', 'log_market_value', 'age', 'age_sq', 'position']].head())

## 3. Exploratory Data Analysis & Visualizations (Saved as PNG and PDF)
Generates publication-quality figures for the `figures/` directory.

In [ ]:
if 'df' in locals():
    # 1. Market Value Distribution (Raw vs Log)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.histplot(df['market_value_eur'] / 1e6, kde=True, ax=axes[0], color='navy')
    axes[0].set_title('Raw Market Value Distribution (M€)')
    axes[0].set_xlabel('Market Value (Million €)')
    
    sns.histplot(df['log_market_value'], kde=True, ax=axes[1], color='darkgreen')
    axes[1].set_title('Log-Transformed Market Value Distribution')
    axes[1].set_xlabel('ln(Market Value)')
    
    plt.tight_layout()
    plt.savefig('figures/mv_distribution.png', dpi=300, bbox_inches='tight')
    plt.savefig('figures/mv_distribution.pdf', bbox_inches='tight')
    plt.close()
    
    # 2. Position Boxplot (RQ1)
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x='position', y='log_market_value', palette='Set2')
    plt.title('Log Market Value by Positional Group')
    plt.xlabel('Position Group')
    plt.ylabel('ln(Market Value)')
    plt.savefig('figures/position_boxplot.png', dpi=300, bbox_inches='tight')
    plt.savefig('figures/position_boxplot.pdf', bbox_inches='tight')
    plt.close()
    
    # 3. Parabolic Age Curve (RQ2)
    plt.figure(figsize=(8, 5))
    sns.regplot(data=df, x='age', y='log_market_value', order=2, scatter_kws={'alpha': 0.3}, line_kws={'color': 'red'})
    plt.title('Parabolic Age Trajectory of Player Valuations')
    plt.xlabel('Age (Years)')
    plt.ylabel('ln(Market Value)')
    plt.savefig('figures/age_valuation_curves.png', dpi=300, bbox_inches='tight')
    plt.savefig('figures/age_valuation_curves.pdf', bbox_inches='tight')
    plt.close()
    
    print("Figures successfully generated and saved in both .png and .pdf formats.")

## 4. Econometric Regression Analysis
- **Model 1 (Baseline):** Age + Age^2 + Control Variables (minutes played, contract years)
- **Model 2 (Positional):** Model 1 + Position Dummies (Defender = Reference group)
- **Model 3 (Full Model):** Model 2 + Positional Metrics (xG per 90, Progressive Passes per 90, Tackles per 90)

In [ ]:
if 'df' in locals():
    # Baseline OLS
    formula_base = 'log_market_value ~ age + age_sq + minutes_played + contract_years'
    model_base = smf.ols(formula_base, data=df).fit(cov_type='HC1')
    
    # Positional OLS
    formula_pos = 'log_market_value ~ age + age_sq + minutes_played + contract_years + C(position, Treatment(reference="Defender"))'
    model_pos = smf.ols(formula_pos, data=df).fit(cov_type='HC1')
    
    # Full Model OLS
    formula_full = 'log_market_value ~ age + age_sq + minutes_played + contract_years + C(position, Treatment(reference="Defender")) + xg_per90 + prog_passes_per90 + tackles_per90'
    model_full = smf.ols(formula_full, data=df).fit(cov_type='HC1')
    
    print("=== FULL MODEL SUMMARY (HC1 Robust Standard Errors) ===")
    print(model_full.summary())

## 5. Diagnostic Tests & Robustness Checks
1. **Breusch-Pagan Test:** Checking for heteroskedasticity.
2. **Multicollinearity Test:** Variance Inflation Factor (VIF < 5.0).

In [ ]:
if 'df' in locals() and 'model_full' in locals():
    # Breusch-Pagan Test
    bp_test = het_breuschpagan(model_full.resid, model_full.model.exog)
    labels = ['LM Statistic', 'LM p-value', 'F Statistic', 'F p-value']
    print("=== Breusch-Pagan Test Results ===")
    for name, value in zip(labels, bp_test):
        print(f"{name}: {value:.6f}")
    
    # VIF Analysis
    X = model_full.model.exog
    vif_data = pd.DataFrame()
    vif_data["feature"] = model_full.model.exog_names
    vif_data["VIF"] = [variance_inflation_factor(X, i) for i in range(X.shape[1])]
    print("\n=== VIF Analysis ===")
    print(vif_data)

## 6. Export Summary Metrics (`data/processed/metrics_summary.csv`)

In [ ]:
if 'df' in locals():
    summary_stats = df.groupby('position').agg(
        count=('market_value_eur', 'count'),
        mean_val=('market_value_eur', 'mean'),
        median_val=('market_value_eur', 'median'),
        mean_age=('age', 'mean')
    ).reset_index()
    
    summary_stats.to_csv('data/processed/metrics_summary.csv', index=False)
    print("Summary metrics exported to data/processed/metrics_summary.csv")